# Deterministic baseline versus 10% hard-negative sampling

Restart the kernel and review the opt-in execution flags before running cells. Only two configurations are enabled, each with seeds 67, 71 and 79 (six fresh runs total). Training is disabled by default; explicitly enable it only when ready. Both use six frames, seven channels, the original 64-channel GRU, batch 16, and up to 40,000 updates with the same early stopping. Prior experiments remain in their old directories. Completed matching new runs are skipped; interrupted runs are preserved in retry folders.

Strict deterministic execution is enabled before model initialization and inside training. Unsupported deterministic operations will raise rather than silently fall back. This improves same-device repeatability, not guarantees across hardware or framework versions. Calibration and replay remain disabled until the paired results have been reviewed.


In [ ]:
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
import sys, json
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src/data_processing").exists())
if str(PROJECT_ROOT / "src") not in sys.path: sys.path.insert(0, str(PROJECT_ROOT / "src"))
from arrival.data import (DataConfig, FrameStore, LabelStore, ArrivalDataset, prepare,
                          fixed_locations, CLASSES, CHANNELS, fingerprint)
from arrival.models import ArrivalNet
from arrival.training import (seed_everything, fit, predict, load_model, metrics,
                              fit_calibrator, calibrated_probs, persistence, motion_baseline)
from arrival.plots import plot_history, plot_evaluation, plot_example
SEED = 67
seed_everything(SEED,deterministic=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE, "| torch:", torch.__version__)
print("Channel order:", CHANNELS)

## Data contract and leakage controls

- Source radar uses the exact 33-color decoder; values are ordinal, **not mm/hour**. The existing conservative radar cleaner remains; no blanket removal of 1–5-pixel targets.
- Crops are `[T, 7, 64, 64]`. The target is pixel `[32,32]`; boundary locations without a full crop are excluded. Channel order is radar, temperature, humidity, wind_u, wind_v, station mask, distance. No explicit coordinates are input, but environmental fields can still reveal geography.
- Currently dry means fewer than five of the center 25 pixels exceed 0.01. First future qualifying observation maps to class 0–11; class 12 requires **all twelve** future observations to remain dry. Missing future data never means no rain.
- For T=6, the required interval is t−25 to t+60 (18 frames, 85 minutes). Shared manifests conservatively reserve T=12 history: t−55 to t+60, keeping anchors identical across history ablations.
- Whole days are partitioned chronologically before constructing anchors; complete windows remain inside one partition. Development ends before the previous held-out period. Train/validation/calibration are separate; a **new** final-test period is disabled until explicitly configured.
- Weather uses the exact frame-time CSV and rejects observation timestamps later than that frame. Historical publication/ingestion timestamps are unavailable, so this is observation-time causality, not measured historical availability.
- Training selects one of four arrival groups with equal probability globally, then an eligible training anchor and location. Expected group frequencies are 25% each; individual batches vary. Validation/calibration locations are fixed and sampled without arrival balancing. Spatial/event correlation remains.

In [ ]:
# Shared across both notebooks. Use a NEW directory when changing label/split definitions.
DATA = DataConfig(crop=64, max_history=12, future=12, patch=5,
                  rain_threshold=0.01, coverage=0.20, stride=1,
                  development_end="2026-08-26", test_start=None, seed=SEED)
DATA_DIR = PROJECT_ROOT / "models/arrival/data_v1"
store = FrameStore(PROJECT_ROOT)
label_store = LabelStore(store, DATA)
# First run may take time: exact source decoding, labels and causal weather caches.
# No API or SQL access. No existing radar-frame checkpoints/caches are overwritten.
prepared = prepare(store, DATA, DATA_DIR, normalization_frames=512)
manifests, norm = prepared["manifests"], prepared["norm"]
print({split: len(anchors) for split, anchors in manifests.items()})
print(prepared["caveat"])
print("Manifest hash:", prepared["manifest_hash"])

In [ ]:
# Seeded uniform anchor subset and natural-frequency spatial sampling for quick evaluation.
# Set EVALUATION_ANCHORS=None for all eligible anchors. Keep this fixed across experiments.
EVALUATION_ANCHORS = 256
EVALUATION_LOCATIONS = 16
def evaluation_records(split):
    anchors = manifests[split]
    if EVALUATION_ANCHORS is not None and len(anchors) > EVALUATION_ANCHORS:
        rng = np.random.default_rng(SEED)
        anchors = sorted(rng.choice(anchors, EVALUATION_ANCHORS, replace=False).tolist())
    return fixed_locations(label_store, anchors, EVALUATION_LOCATIONS, SEED)
validation_records = evaluation_records("validation")
calibration_records = evaluation_records("calibration")
(DATA_DIR / "evaluation_locations.json").write_text(json.dumps({
    "validation": validation_records, "calibration": calibration_records}, indent=2))
print("Fixed validation/calibration examples:", len(validation_records), len(calibration_records))
def make_dataset(history=6, channels=tuple(range(7)), split="train"):
    records = None if split == "train" else (validation_records if split == "validation" else calibration_records)
    return ArrivalDataset(store, label_store, manifests[split], norm, history, channels,
                          per_anchor=32, seed=SEED, records=records)

## Experiment matrix and controls

All runs use the same source normalization, balanced training sampler, fixed natural-frequency validation locations, update budget and model-selection criterion. `max_history=12` keeps anchors eligible for every history condition. Compare compute time as well as validation skill; fewer recurrent parameters do not guarantee faster or better forecasts.

Start with CNN versus GRU. Enable one family of ablations at a time. Repeat promising comparisons across seeds before treating small differences as meaningful. A cropped local model may lack upstream context for long-range arrivals; crop-size/multiscale context is a follow-on study, requiring a new shared dataset definition.

In [ ]:
EXPERIMENTS = [
    dict(name="gru_deterministic_baseline",kind="gru",history=6,channels=tuple(range(7)),output="categorical",hard_negative_fraction=0.0),
    dict(name="gru_deterministic_hard10",kind="gru",history=6,channels=tuple(range(7)),output="categorical",hard_negative_fraction=0.1),
]
display(pd.DataFrame(EXPERIMENTS))
RUN_EXPERIMENTS = False
ENABLED = {e["name"] for e in EXPERIMENTS}
SEEDS = [67, 71, 79]
UPDATE_BUDGET = 40000
SWEEP_DIR = PROJECT_ROOT / "models/arrival/experiments_deterministic_v1"


In [ ]:
if RUN_EXPERIMENTS:
    SWEEP_DIR.mkdir(parents=True,exist_ok=True)
    for experiment in EXPERIMENTS:
        if experiment["name"] not in ENABLED: continue
        for seed in SEEDS:
            from arrival.run_paths import sweep_run_path, write_completed_result
            path = sweep_run_path(SWEEP_DIR / f"{experiment['name']}_seed{seed}")
            if (path / "result.json").exists():
                completed = json.loads((path / "result.json").read_text())
                if (not completed.get("deterministic",False) or completed.get("update_budget") != UPDATE_BUDGET or
                    completed["manifest_hash"] != prepared["manifest_hash"] or
                    completed["validation_records_hash"] != fingerprint(validation_records) or
                    json.dumps(completed["experiment"],sort_keys=True) != json.dumps(experiment,sort_keys=True)):
                    raise ValueError(f"Completed run configuration differs: {path}")
                print("Skipping completed run:", path.name)
                continue
            seed_everything(seed,deterministic=True)
            training = make_dataset(experiment["history"],experiment["channels"])
            training.seed = seed
            training.hard_negative_fraction = experiment.get("hard_negative_fraction",0.0)
            training.hard_negative_warmup = experiment.get("hard_negative_warmup",0)
            training.hard_negative_ramp = experiment.get("hard_negative_ramp",0)
            validation = make_dataset(experiment["history"],experiment["channels"],"validation")
            candidate = ArrivalNet(history=experiment["history"],channels=len(experiment["channels"]),
                kind=experiment["kind"],output=experiment["output"],
                hidden_dim=experiment.get("hidden_dim",64),pooling=experiment.get("pooling","global"),
                recurrent_layers=experiment.get("recurrent_layers",1),
                encoder_extra_layers=experiment.get("encoder_extra_layers",0)).to(DEVICE)
            parameter_count = sum(p.numel() for p in candidate.parameters())
            if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()
            history = fit(candidate,training,validation,path,budget=UPDATE_BUDGET,
                          eval_every=250,patience=40,batch=16,seed=seed,selection="mean_ap",deterministic=True)
            log_probs, truth = predict(candidate,validation)
            result = metrics(calibrated_probs(log_probs),truth)
            payload = dict(experiment=experiment,seed=seed,parameters=parameter_count,
                sample_counts=training.sample_counts,deterministic=True,update_budget=UPDATE_BUDGET,
                peak_gpu_allocated_mb=torch.cuda.max_memory_allocated()/1024**2 if torch.cuda.is_available() else None,
                actual_updates=history[-1]["step"],elapsed_seconds=history[-1]["elapsed_seconds"],
                validation_metrics=result,manifest_hash=prepared["manifest_hash"],
                validation_records_hash=fingerprint(validation_records))
            np.savez_compressed(path / "validation_predictions.npz",probabilities=calibrated_probs(log_probs),labels=truth.numpy())
            write_completed_result(path, payload)
            del candidate
            if torch.cuda.is_available(): torch.cuda.empty_cache()
else:
    print("Enable an experiment family and set RUN_EXPERIMENTS = False when ready.")

## Compare results without reopening the final test

Compare mean validation average precision and 15/30/60-minute AP, alongside NLL, Brier scores, parameter count, elapsed time and peak GPU memory. The enabled comparisons change one factor: baseline 64-channel GRU with global pooling, 96-channel GRU with global pooling, or 64-channel GRU with a spatial 4x4 pooled head. All use six frames, seven channels, seed 67 and up to 30,000 updates. Training runs sequentially. These are fresh runs, not resumptions of v5. Calibration and notification replay are separate follow-up evaluations after selecting a candidate. The same maximum update budget is used; early stopping may yield different actual updates, which are reported. Validation examples share anchors/events; use day-level uncertainty below rather than pretending every crop is independent.

In [ ]:
from arrival.run_paths import completed_run
results = [json.loads(p.read_text()) for p in sorted(SWEEP_DIR.glob("*/result.json")) if completed_run(p.parent)]
rows = []
for r in results:
    if r["manifest_hash"] != prepared["manifest_hash"] or r["validation_records_hash"] != fingerprint(validation_records):
        raise ValueError("Experiment result uses a different comparison dataset")
    m = r["validation_metrics"]
    rows.append(dict(name=r["experiment"]["name"],seed=r["seed"],parameters=r["parameters"],
        updates=r["actual_updates"],seconds=r["elapsed_seconds"],NLL=m["nll"],macro_F1=m["macro_f1_all_13"],
        peak_gpu_mb=r.get("peak_gpu_allocated_mb"),
        mean_AP=np.mean([m["ranking"][str(h)]["average_precision"] for h in (15,30,60)]),
        **{f"AP_{h}":m["ranking"][str(h)]["average_precision"] for h in (15,30,60)},
        missed_arrivals=m["missed_arrivals"],false_arrivals=m["false_arrivals"],
        **{f"Brier_{h}":m["horizons"][str(h)]["brier"] for h in (15,30,60)}))
comparison = pd.DataFrame(rows)
if len(comparison):
    display(comparison.sort_values("mean_AP",ascending=False))
    comparison.to_csv(SWEEP_DIR / "comparison.csv",index=False)
    comparison.set_index("name")[["Brier_15","Brier_30","Brier_60"]].plot.bar(figsize=(12,4))
    plt.ylabel("Brier score (lower is better)"); plt.tight_layout(); plt.show()
else:
    print("No completed experiment results yet.")

In [ ]:
# Compare paired seeds; do not choose a method from its single best seed.
if len(comparison):
    score_columns = ["mean_AP","AP_15","AP_30","AP_60"]
    display(comparison.groupby("name")[score_columns].agg(["count","mean","std"]))
    paired = comparison.pivot(index="seed",columns="name",values="mean_AP")
    if {"gru_deterministic_baseline","gru_deterministic_hard10"}.issubset(paired.columns):
        paired["hard10_minus_baseline"] = paired["gru_deterministic_hard10"]-paired["gru_deterministic_baseline"]
        display(paired)
        paired.to_csv(SWEEP_DIR/"paired_seed_comparison.csv")
        print("Complete pairs:",paired.dropna().shape[0],"of",len(SEEDS))
    print("Calibration/replay still needed before notification conclusions. Three seeds are not a final independent test.")


In [ ]:
# Optional paired day-block bootstrap for two completed runs; no future labels used.
PAIR = ("gru_deterministic_baseline_seed67", "gru_deterministic_hard10_seed67")
from arrival.run_paths import find_completed_run
paired_runs = [find_completed_run(SWEEP_DIR / name) for name in PAIR]
paths = [p / "validation_predictions.npz" for p in paired_runs if p is not None]
if len(paths)==2 and all(p.exists() for p in paths):
    predictions = [np.load(p) for p in paths]
    assert np.array_equal(predictions[0]["labels"],predictions[1]["labels"])
    days = np.array([r[0][:8] for r in validation_records]); unique_days = np.unique(days)
    y = predictions[0]["labels"] < 6
    errors = [(p["probabilities"][:,:6].sum(1)-y)**2 for p in predictions]
    daily_difference = np.array([(errors[1][days==day]-errors[0][days==day]).mean() for day in unique_days])
    rng = np.random.default_rng(SEED)
    draws = rng.choice(daily_difference,(2000,len(daily_difference)),replace=True).mean(1)
    print("Equal-day mean Brier-30 difference (second minus first):",daily_difference.mean())
    print("Day-bootstrap 95% interval:",np.quantile(draws,[.025,.975]))
    print("Few days and storms spanning multiple days limit this interval; inspect event groups too.")

## Calibrate a selected sampling/architecture candidate

Training is disabled by default. Run setup/data cells and this calibration cell to load the saved winner; no retraining is needed. Fit twelve cumulative logistic curves on the separate calibration partition, then project across horizons to prevent crossings. This is the improved v5 candidate method, refitted to the winner. Calibration-fit scores are in-sample; validation scores are development results because validation selected this model. The optional replay cell below measures arrival notifications offline. Leave final testing disabled.

In [ ]:
SELECTED_RUN = "gru_deterministic_baseline_seed67"  # Compare results before treating this as a winner.
from arrival.run_paths import find_completed_run
selected_path = find_completed_run(SWEEP_DIR / SELECTED_RUN)
RUN_CALIBRATION = False  # Enable only after choosing a successful candidate.
selected = None
calibrator = None
if RUN_CALIBRATION and selected_path is not None and (selected_path / "best.pt").exists():
    selected, state = load_model(selected_path / "best.pt",DEVICE)
    if state["validation_records_hash"] != fingerprint(validation_records) or state["norm"] != norm:
        raise ValueError("Selected checkpoint data contract differs")
    calibration = make_dataset(state["model_config"]["history"],tuple(state["channels"]),"calibration")
    log_probs, truth = predict(selected,calibration)
    from arrival.calibration import fit_projected_calibrator
    calibrator = fit_projected_calibrator(log_probs,truth)
    probabilities = calibrated_probs(log_probs,calibrator)
    result = metrics(probabilities,truth)
    plot_evaluation(probabilities,truth,result)
    (selected_path / "calibrator.json").write_text(json.dumps(calibrator,indent=2))

    import hashlib
    calibrator['checkpoint_sha256'] = hashlib.sha256((selected_path/'best.pt').read_bytes()).hexdigest()
    calibrator['calibration_records_hash'] = fingerprint(calibration_records)
    (selected_path/'calibrator.json').write_text(json.dumps(calibrator,indent=2))
    validation = make_dataset(state['model_config']['history'],tuple(state['channels']),'validation')
    validation_lp, validation_y = predict(selected,validation)
    comparison = {'raw':metrics(calibrated_probs(validation_lp),validation_y),
                  'projected':metrics(calibrated_probs(validation_lp,calibrator),validation_y)}
    (selected_path/'calibration_validation_metrics.json').write_text(json.dumps(comparison,indent=2))
    np.savez_compressed(selected_path/'calibration_predictions.npz',log_probs=log_probs.numpy(),labels=truth.numpy())
    display(pd.DataFrame([dict(method=name,horizon=h,AP=r['average_precision'],
        brier=m['horizons'][int(h)]['brier']) for name,m in comparison.items() for h,r in m['ranking'].items()]))
    plot_evaluation(calibrated_probs(validation_lp,calibrator),validation_y,comparison['projected'])
    print('Winner calibration ready for offline notification replay:',selected_path)

else:
    print("Complete/select a run first.")

In [ ]:
# Run after the winner calibration cell; offline only, no notifications sent.
RUN_NOTIFICATION_REPLAY = False  # Set True to measure the new winner's alert performance.
if RUN_NOTIFICATION_REPLAY:
    import subprocess, sys
    if globals().get('selected_path') is None:
        raise RuntimeError('Run the selection/calibration cell first and select a completed run.')
    if not (selected_path / 'calibrator.json').is_file():
        raise RuntimeError(f'No saved calibration for {selected_path.name}. Set RUN_CALIBRATION = False in the preceding cell, run it to completion, then retry replay. No retraining is needed.')
    subprocess.run([sys.executable,str(PROJECT_ROOT/'scripts/replay_arrival_notifications.py'),
                    '--run',str(selected_path),'--calibration',str(selected_path/'calibrator.json'),
                    '--output',str(PROJECT_ROOT/'reports'/f'arrival_replay_{selected_path.name}')],
                   cwd=PROJECT_ROOT,check=True)


In [ ]:
RUN_FINAL_TEST = False
if RUN_FINAL_TEST:
    if selected is None or calibrator is None or not DATA.test_start or not manifests["test"]:
        raise RuntimeError("Require a frozen selected model, calibrator and new untouched test period")
    test_records = fixed_locations(label_store,manifests["test"],EVALUATION_LOCATIONS,SEED)
    test_data = ArrivalDataset(store,label_store,manifests["test"],norm,
        history=state["model_config"]["history"],channels=tuple(state["channels"]),records=test_records)
    lp, yt = predict(selected,test_data)
    final_probabilities = calibrated_probs(lp,calibrator)
    final_metrics = metrics(final_probabilities,yt)
    (selected_path / "final_test_metrics.json").write_text(json.dumps(final_metrics,indent=2))
    plot_evaluation(final_probabilities,yt,final_metrics)
if selected is not None and calibrator is not None:
    export = dict(model_config=state["model_config"],channels=state["channels"],normalization=norm,
        data_config=vars(DATA),classes=CLASSES,calibrator=calibrator,manifest_hash=prepared["manifest_hash"])
    (selected_path / "inference_config.json").write_text(json.dumps(export,indent=2))

## Explicitly deferred

No architecture/loss result is assumed superior in advance. Neither notebook deploys to Telegram, generates predicted radar images, changes rain-ending rules, or uses the old inspected test set to select a model. Dense optical flow and crop-size/multiscale context are follow-on experiments if the global translation baseline and history ablations identify a need.